# DDPE Spark Connect + AIDP Governance

Interactive queries against the Dell Data Processing Engine (DDPE) Spark Connect
server `jirawut-demo`, and a side-by-side comparison with governed access through
AIDP/Starburst BIAC.

**What this notebook shows:**
1. How to connect to DDPE Spark Connect from a notebook.
2. The identity Spark actually executes as (`current_user()`).
3. What the Spark engine can and cannot reach - and why BIAC roles/masks/row
   filters do **not** apply to direct storage access.
4. The governed path: the same data queried through Trino where BIAC enforces
   identity, masking and row filters.

Prereq (local dev host only): `bash scripts/start_spark_proxy.sh` once - it
creates a local TLS-terminating proxy at `127.0.0.1:15003` because the DDPE
ingress cert is signed by an internal CA not in our trust bundle and the istio
gateway routes on the gRPC `:authority` header. Inside the managed DDPE
JupyterLab image the cluster CA is baked in, so connect directly instead.

In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path('/home/kiran/aidp-governance-showcase')
sys.path.insert(0, str(PROJECT / 'scripts'))
os.chdir(PROJECT)  # avoid module-shadowing issues in odd cwds

from aidp_common import load_settings
settings = load_settings()

DDPE_HOST  = os.environ.get('DDPE_GRPC_HOST', 'jirawut-demo-grpc.ddpe.lab9bgp.com')
DDPE_TOKEN = os.environ.get('DDPE_SPARK_TOKEN')
PROXY_PORT = os.environ.get('DDPE_PROXY_PORT', '15003')
CA_BUNDLE  = os.environ.get('DDPE_CA_BUNDLE', '/tmp/sparktls/ca.pem')
assert DDPE_TOKEN, 'set DDPE_SPARK_TOKEN in .env or the environment'
os.environ['GRPC_DEFAULT_SSL_ROOTS_FILE_PATH'] = CA_BUNDLE
print('endpoint :', DDPE_HOST)
print('proxy    : 127.0.0.1:' + PROXY_PORT)
print('ca       :', CA_BUNDLE)

## 1. Connect via Spark Connect

`grpc.default_authority` is the key option: the istio gateway routes on the
gRPC `:authority` pseudo-header, so even through the local proxy we must send
the real DDPE hostname. `sc://localhost` alone gets `UNIMPLEMENTED` on every
method.

In [ ]:
from pyspark.sql.connect.client.core import ChannelBuilder
from pyspark.sql.connect.session import SparkSession as RemoteSession

def connect(local_proxy=True):
    target = f'127.0.0.1:{PROXY_PORT}' if local_proxy else f'{DDPE_HOST}:443'
    cb = ChannelBuilder(
        f'sc://{target}/;token={DDPE_TOKEN}',
        channelOptions=[('grpc.default_authority', DDPE_HOST)])
    return RemoteSession(connection=cb)

try:
    spark = connect(local_proxy=True)
    print('via proxy :', spark.sql('SELECT 1 AS ok').collect())
except Exception as e:
    print('proxy failed (', str(e)[:80], ') - trying direct')
    spark = connect(local_proxy=False)
    print('direct    :', spark.sql('SELECT 1 AS ok').collect())

## 2. Who does Spark execute as?

The engine runs as its own service identity - **not** as a Keycloak user. Any
role/persona distinction from the governance demo simply does not exist here.

In [ ]:
print('current_user :', spark.sql('SELECT current_user()').collect())
print('spark master :', spark.sql('SET spark.master').collect())
print('catalogs     :', spark.sql('SHOW CATALOGS').collect())
print('namespaces   :', spark.sql('SHOW NAMESPACES').collect())
print('s3a endpoint :', spark.sql('SET spark.hadoop.fs.s3a.endpoint').collect())
print('s3a creds    :', spark.sql('SET spark.hadoop.fs.s3a.access.key').collect())

## 3. Try to read the governed Iceberg table

The demo data lives in `s3a://js-demo/warehouse/gov_demo` (Iceberg, MinIO).
This instance was launched **without** working S3 credentials for that path -
its baked-in creds target the AIDP `s3Proxy` endpoint - so the read fails.
Even if it succeeded, the engine would return **raw unmasked data with no
audit trail**: BIAC is a SEP/Trino-side control and never sees storage reads.

In [ ]:
# register an Iceberg hadoop catalog against the same warehouse
for kv in [
    'spark.sql.catalog.ice=org.apache.iceberg.spark.SparkCatalog',
    'spark.sql.catalog.ice.type=hadoop',
    'spark.sql.catalog.ice.warehouse=s3a://js-demo/warehouse',
]:
    spark.sql('SET ' + kv).collect()

try:
    rows = spark.sql(
        'SELECT customer_name, national_id, country_code '
        'FROM ice.gov_demo.customer_transactions LIMIT 5').collect()
    print('READ SUCCEEDED - raw, unmasked, ungoverned:', rows)
except Exception as e:
    print('read failed as launched:', str(e)[:300])
    print()
    print('(with valid s3a creds baked into the instance, this read returns')
    print(' RAW data - no BIAC role checks, masks, row filters or audit)')

## 4. The governed path - same data through AIDP/Starburst

For comparison, query `gov_demo.customer_transactions` through Trino as
`gov-tha` (the Thailand analyst). BIAC applies the TH-only row filter and
strong column masks - evaluated inside the engine, audited in
`/api/v1/biac/audit/accessLogs`.

In [ ]:
from aidp_common import run_sql

cols, rows = run_sql(
    settings, 'gov-tha', settings.demo_user_password,
    'SELECT customer_name, national_id, country_code, risk_score '
    'FROM js_financial_ice.gov_demo.customer_transactions '
    'ORDER BY transaction_id LIMIT 8')
for r in rows:
    print(r)
print()
print('via Trino as gov-tha -> only TH rows, all PII masked (BIAC-enforced)')

## Takeaways

| Path | Identity | Enforcement |
| --- | --- | --- |
| Trino / DDAE | real Keycloak user -> group -> BIAC role | grants, masks, row filters, audit |
| DDPE Spark Connect | `spark` (engine service) | none inside SEP - only storage-level |

- DDPE instances read object storage directly. If a Spark instance is launched
  with valid `fs.s3a.*` credentials for the governed bucket, it bypasses all
  BIAC policies. Treat storage credentials as the real perimeter.
- To keep governance for Spark workloads, route data access through Trino
  (JDBC) or the AIDP `s3Proxy` where AIDP controls the path.
- The `sc://<host>-grpc.<domain>/;token=` URI is documented in Dell's DDLH
  admin guide; the token is short-lived - regenerate via
  `dell-data-processing-engine instance status name=<instance>`.